# Resume training on Colab A100

**Pre-requisites** (set once before running this notebook):
1. Run `prep_data_hub.py` locally to upload `train/val/test.parquet` to a HF dataset repo.
2. From the most recent training pod, push the latest checkpoint dir to a HF model repo (e.g. `junho5400/svg-finetune-qwen-7b-lora-resume`) using `hf upload <repo> outputs/checkpoint/checkpoint-XXXX --include '*'`.

**This notebook**: Runtime → A100, then Run all. Each cell sets up env, downloads data + checkpoint, runs training, pushes the new checkpoint at end. Designed to fit in one ~10-12 hour Colab Pro session.

**For multi-session training**: the LATEST cell pushes the new checkpoint to `junho5400/svg-finetune-qwen-7b-lora-resume` (overwriting the prior one). Next session, just open this notebook again — it'll pull the new checkpoint and continue.

In [ ]:
# === Config — edit these to match your repos ===
GIT_REPO = 'https://github.com/junho5400/svg_finetune.git'
DATA_REPO = 'junho5400/svg-finetune-data'             # HF dataset repo
RESUME_REPO = 'junho5400/svg-finetune-qwen-7b-lora-resume'  # HF model repo with latest checkpoint
WANDB_ENTITY = 'junho5400-northwestern-university'
MAX_STEPS = 30000  # session budget — increase if you want longer; A100 typically does ~3000 steps/hour

In [ ]:
# === Auth (paste your tokens) ===
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('HF write token: ')
os.environ['WANDB_API_KEY'] = getpass('WandB API key: ')
os.environ['WANDB_ENTITY'] = WANDB_ENTITY

In [ ]:
# === Clone repo + install pinned deps ===
!cd /content && git clone {GIT_REPO} svg_finetune || (cd svg_finetune && git pull)
!cd /content/svg_finetune && pip install -q -r requirements.txt

In [ ]:
# === Pull data parquets from HF dataset repo ===
from huggingface_hub import hf_hub_download, snapshot_download
import os

DATA_DIR = '/content/svg_finetune/data'
os.makedirs(DATA_DIR, exist_ok=True)
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    print(f'fetching {fname}...')
    hf_hub_download(repo_id=DATA_REPO, filename=fname,
                    repo_type='dataset', local_dir=DATA_DIR)
print('data ready')

In [ ]:
# === Pull latest checkpoint ===
OUT_CKPT_DIR = '/content/svg_finetune/outputs/checkpoint'
os.makedirs(OUT_CKPT_DIR, exist_ok=True)
# Snapshot the whole resume repo into a subfolder named like a checkpoint dir
ckpt_path = snapshot_download(repo_id=RESUME_REPO, local_dir=f'{OUT_CKPT_DIR}/checkpoint-resume')
print(f'checkpoint at: {ckpt_path}')
!ls -lh {ckpt_path}

In [ ]:
# === Tweak config for this session: bigger batch (no grad-ckpt needed on A100), longer max_length if memory allows ===
# A100 80GB can fit batch=4 max_length=2048 without gradient checkpointing.
# A100 40GB stay at batch=2 max_length=1024 to be safe.
# Adjust if you have 80GB.
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)')

# Inject MAX_STEPS into the training run via an env override (cleaner than editing config.py)
os.environ['SVG_MAX_STEPS_OVERRIDE'] = str(MAX_STEPS)
print(f'will train up to step {MAX_STEPS} (will resume from latest checkpoint)')

In [ ]:
# === Run training ===
# Auto-resume kicks in because outputs/checkpoint/checkpoint-resume/ exists.
!cd /content/svg_finetune && python train.py 2>&1 | tee train.log

In [ ]:
# === Push new latest checkpoint to HF Hub for next session's resume ===
import glob
checkpoints = sorted(glob.glob(f'{OUT_CKPT_DIR}/checkpoint-*'))
# Filter out 'checkpoint-resume' (the one we downloaded) — we want the newer one
newer = [c for c in checkpoints if 'resume' not in c]
if not newer:
    print('no new checkpoint produced — run probably failed early. Check train.log above.')
else:
    latest = newer[-1]
    print(f'latest new checkpoint: {latest}')
    print(f'pushing to {RESUME_REPO} (overwriting prior resume point)...')
    !cd {latest} && hf upload {RESUME_REPO} . --include '*'

In [ ]:
# === Optional: inspect WandB run + step we reached ===
!tail -30 /content/svg_finetune/train.log